# Phase 12 - Sonde II: derselbe Test, richtig dimensioniert

**Braucht eine A100**, ~20 min.

Der erste Sondenlauf lieferte das vorregistrierte Verdikt `KEINE-VORHERSAGE`: H1 auf
der Einbettung p = 0.061 gegen die Schwelle 0.05, L11 p = 0.0195 gegen 0.0167 nach
Bonferroni. Knapp daneben ist daneben.

Die Nachlese hat gezeigt **warum**. Ueber 200 zufaellige Teilungen derselben Groesse
liegt der historische Schnitt bei der Einbettung im **47. Perzentil** - exakt in der
Mitte. Der Schnitt war nicht ungluecklich; 26 Trainingswoerter sind schlicht zu wenig.
Dazu: mit einem fuer Statistik *und* Null identischen alpha wird H2 staerker, nicht
schwaecher (emb p = 0.0075, L11 0.0020, L23 0.0015, L35 0.0010), ohne das staerkste
Wort bleibt alles stehen, und der triviale Merkmalssatz bleibt flach.

Diese Zelle wiederholt den Test mit der Trainingsgroesse, die er gebraucht haette:
**angepasst auf alle 64 schon gemessenen Woerter, geprueft an 24 neuen.** Kein neuer
Sweep, keine neue Handkodierung. Die 64 Trainingsraten stammen aus dem ersten Lauf und
werden hier nicht neu erhoben - sie sind Messung, nicht Wahl.

## Die Auswahl der Darstellung ist jetzt sauber

`L23` war im ersten Lauf die staerkste (R² 0.29 gegen 0.05 fuer die reine Einbettung).
Sie wurde also auf **Lauf A ausgewaehlt** und wird hier auf **Lauf B geprueft** - die
zulaessige Reihenfolge, im Gegensatz zu Auswahl und Bewertung auf denselben Daten.

* **P1 (primaer)** L23, angepasst an 64, sagt 24 neue mit positivem Spearman voraus
* **P2 (sekundaer)** dasselbe fuer die Einbettung
* **P3 (Kontrolle)** Wortlaenge und Tokenzahl duerfen *nicht* vorhersagen
* **Tor** vier Anker (`native`, `corresponding`, `exact`, `precise`) pruefen, ob dieser
  Lauf die Skala des Vorlaufs ueberhaupt reproduziert

Die 24 neuen Woerter sind bewusst unauffaellig gewaehlt - keines soll ein sicherer
Extremwert sein, sonst waere die Rangkorrelation geschenkt.

Faellt P1 auch hier, ist der Befund **belastbar negativ**: dann ist die Kipprate keine
Funktion der Darstellung, die sich mit dieser Datenmenge finden laesst, und die
Spreizung von 0 bis 79 % braucht eine ganz andere Erklaerung.


In [ ]:
# === PHASE 12 - SONDE II: DERSELBE TEST, RICHTIG DIMENSIONIERT =============
# Der erste Sondenlauf hat das vorregistrierte Verdikt KEINE-VORHERSAGE
# geliefert: H1 auf der Einbettung p=0.061 gegen die Schwelle 0.05, L11
# p=0.0195 gegen 0.0167 nach Bonferroni. Knapp daneben ist daneben.
#
# Die Nachlese hat gezeigt, WARUM. Ueber 200 zufaellige Teilungen derselben
# Groesse liegt der historische Schnitt ALT->NEU bei der Einbettung im 47.
# Perzentil - also exakt in der Mitte. Der Schnitt war nicht ungluecklich,
# 26 Trainingswoerter sind schlicht zu wenig. Ausserdem: mit einem fuer
# Statistik UND Null identischen alpha wird H2 staerker, nicht schwaecher
# (emb p=0.0075, L11 0.0020, L23 0.0015, L35 0.0010), und ohne das staerkste
# Wort bleibt alles stehen. Der triviale Merkmalssatz bleibt flach.
#
# DIESE ZELLE WIEDERHOLT DEN TEST MIT DER TRAININGSGROESSE, DIE ER GEBRAUCHT
# HAETTE: angepasst auf ALLE 64 schon gemessenen Woerter, geprueft an 24
# NEUEN. Kein neuer Sweep, keine neue Handkodierung.
#
# DIE AUSWAHL DER DARSTELLUNG IST JETZT SAUBER VORREGISTRIERT:
#   L23 war im ersten Lauf die staerkste (R2 0.29 gegen 0.05 fuer die reine
#   Einbettung). Sie wurde also auf Lauf A AUSGEWAEHLT und wird hier auf
#   Lauf B GEPRUEFT - das ist die zulaessige Reihenfolge, im Gegensatz zur
#   Auswahl und Bewertung auf denselben Daten.
#   P1 primaer: L23. P2 sekundaer: emb, unveraendert.
#
# VORAB REGISTRIERT:
#   P1 (primaer)  Ridge auf L23, angepasst an die 64 historischen Woerter,
#                 sagt die 24 neuen mit positivem Spearman voraus.
#                 Permutationstest (Trainingsziele gemischt, voll neu
#                 angepasst), alpha 0.05.
#   P2            dasselbe fuer emb - sekundaer, beschreibend.
#   P3 (Kontrolle) Wortlaenge und Tokenzahl sagen NICHT voraus.
#   TOR           Vier Anker (native, corresponding, exact, precise) laufen
#                 mit. native muss der hoechste, precise der niedrigste der
#                 vier sein, sonst ist die Skala nicht vergleichbar und P1
#                 nicht deutbar.
# Faellt P1 auch hier, ist der Befund belastbar negativ: die Kipprate ist
# dann keine Funktion der Darstellung, die sich mit 64 Beispielen finden
# laesst. Das ist ein vorgesehener Ausgang.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, unicodedata, random
import numpy as np, glob, json, gc, sys, time
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError("GPU nicht leer genug (%.1f GB frei, ~45 noetig). "
                       "Laufzeit -> Sitzung neu starten, dann NUR diese Zelle."%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN","phase12_sonde2")
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
# ---------------- reine Logik (offline geprueft) ----------------------------
PHRASE="each service's local name"
# HISTORISCH: Treffer aus dem ersten Sondenlauf, je 48 Ziehungen, breite Rate.
# Diese Zahlen sind Messung, keine Wahl - sie werden hier nicht neu erhoben.
HIST={"precise":0,"particular":0,"specific":2,"designated":5,"exact":3,"applicable":6,
      "actual":15,"correct":13,"given":16,"individual":27,"relevant":14,"respective":28,
      "proper":18,"corresponding":20,"equivalent":0,"canonical":2,"customary":12,
      "matching":15,"standard":9,"associated":15,"verbatim":12,"printed":13,
      "written":22,"literal":20,"own":19,"native":38,"usual":5,"common":0,"typical":4,
      "normal":6,"regular":24,"ordinary":3,"general":8,"main":10,"primary":13,
      "principal":13,"current":27,"existing":3,"established":4,"recognized":14,
      "accepted":5,"preferred":1,"assigned":3,"listed":15,"stated":11,"displayed":11,
      "published":17,"registered":4,"formal":9,"popular":0,"familiar":0,"everyday":0,
      "traditional":13,"modern":0,"alternative":2,"secondary":8,"short":0,
      "abbreviated":0,"complete":0,"true":14,"real":11,"genuine":12,"authentic":7,
      "appropriate":14}
N_HIST=48
# 24 NEUE Woerter. Bewusst unauffaellig gewaehlt - keines soll ein sicherer
# Extremwert sein, sonst waere die Rangkorrelation geschenkt.
NEU=["apparent","approved","assumed","commercial","conventional","corporate",
     "default","distinct","exclusive","external","full","historical","intended",
     "internal","known","legal","public","separate","simple","suitable","unique",
     "universal","valid","verified"]
ANKER=["native","corresponding","exact","precise"]     # Tor, nicht Test
def phrase_mit(adj):
    return PHRASE if not adj else "each service's %s local name"%adj
def setze_arm(text,neu):
    if text.count(PHRASE)!=1: return text,False
    return text.replace(PHRASE,neu),True
def logit(p,eps=1e-6):
    p=min(max(p,eps),1-eps); return math.log(p/(1-p))
def ziel(k,n): return logit((k+0.5)/(n+1.0))
def ridge_fit(X,y,alpha):
    mx=X.mean(0); my=float(y.mean()); Xc=X-mx
    return dict(mx=mx,my=my,Xc=Xc,
                a=np.linalg.solve(Xc@Xc.T+alpha*np.eye(len(y)),y-my))
def ridge_pred(m,Xn): return (Xn-m["mx"])@m["Xc"].T@m["a"]+m["my"]
def waehle_alpha(X,y,alphas):
    best=(None,float("inf"))
    for al in alphas:
        s=0.0
        for i in range(len(y)):
            tr=[j for j in range(len(y)) if j!=i]
            s+=(ridge_pred(ridge_fit(X[tr],y[tr],al),X[i:i+1])[0]-y[i])**2
        if s<best[1]: best=(al,s)
    return best[0]
def raenge(v):
    idx=sorted(range(len(v)),key=lambda i:v[i]); r=[0.0]*len(v); i=0
    while i<len(idx):
        j=i
        while j+1<len(idx) and v[idx[j+1]]==v[idx[i]]: j+=1
        m=(i+j)/2.0+1.0
        for k in range(i,j+1): r[idx[k]]=m
        i=j+1
    return r
def pearson(x,y):
    n=len(x); mx=sum(x)/n; my=sum(y)/n
    sxy=sum((a-mx)*(b-my) for a,b in zip(x,y))
    sx=math.sqrt(sum((a-mx)**2 for a in x)); sy=math.sqrt(sum((b-my)**2 for b in y))
    return sxy/(sx*sy) if sx*sy else float("nan")
def spearman(x,y): return pearson(raenge(list(x)),raenge(list(y)))
def bestimmtheit(y,yh):
    y=np.asarray(y,float); yh=np.asarray(yh,float)
    ss=float(((y-yh)**2).sum()); st=float(((y-y.mean())**2).sum())
    return 1.0-ss/st if st>0 else float("nan")
def haltefeld(Xa,ya,Xn,yn,alphas,perm=4000,startwert=20260805):
    """Anpassen auf die historischen Woerter, vorhersagen auf die neuen.
       Der Nulltest mischt die TRAININGS-Ziele und passt vollstaendig neu an."""
    al=waehle_alpha(Xa,ya,alphas)
    yh=ridge_pred(ridge_fit(Xa,ya,al),Xn)
    rho=spearman(yh,yn)
    rnd=random.Random(startwert); yy=list(ya); tr=0
    for _ in range(perm):
        rnd.shuffle(yy)
        if spearman(ridge_pred(ridge_fit(Xa,np.array(yy),al),Xn),yn)>=rho: tr+=1
    return dict(alpha=float(al),rho=float(rho),p=(tr+1)/(perm+1.0),
                r2=float(bestimmtheit(yn,yh)),vorhersage=[float(v) for v in yh])
def pruefe_tor(rate):
    """rate: Anker -> gemessene Rate in DIESEM Lauf. Die Rangfolge der vier
       muss die des Vorlaufs wiederholen, sonst ist die Skala verschoben."""
    if not all(a in rate for a in ANKER): return None,[]
    # bindungstolerant: 'precise' und 'exact' lagen im Vorlauf bei 0.0% und
    # 6.2% - liegen beide in diesem Lauf bei null, waere ein strenges
    # "precise ist das Minimum" ein Fehlalarm der Sortierreihenfolge.
    alt=[HIST[a]/N_HIST for a in ANKER]; neu=[rate[a] for a in ANKER]
    rho=spearman(alt,neu)
    ok=bool(rate["native"]>max(rate[a] for a in ANKER if a!="native")
            and rate["precise"]<=min(rate["exact"],rate["corresponding"])
            and rho>0.5)
    return ok,[(a,rate[a],HIST[a]/N_HIST) for a in ANKER]+[("spearman",rho,1.0)]
def urteil_sonde2(tor_ok,P1,P2,P3,alpha=0.05):
    if tor_ok is False: return "NICHT-VERGLEICHBAR"
    if P3 is not None and P1 is not None and P3["rho"]>0 and P3["p"]<alpha \
       and P3["rho"]>=P1["rho"]: return "TRIVIAL-REICHT"
    if P1 is not None and P1["rho"]>0 and P1["p"]<alpha: return "BESTAETIGT"
    if P2 is not None and P2["rho"]>0 and P2["p"]<alpha: return "NUR-EINBETTUNG"
    return "WIDERLEGT"
def wilson(k,n,z=1.96):
    if n==0: return (0.,0.,0.)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n); h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),
     (0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que "
        "sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will "
        "would can it on as at be by".split())
PTES=set("nome nomes servico servicos armazenamento limite limites preco mes gratuito "
         "conta cada para com uma nao mais seu sua nombre servicio servicios "
         "almacenamiento precio cuenta los las del con mas su".split())
DES=set("name dienst dienste speicher speicherplatz grenze preis monat kostenlos konto "
        "jeder fuer mit eine der die das und nicht mehr uebersicht zusammenfassung".split())
def _fremd(s):
    return [c for c in s if c.isalpha() and ord(c)>=0x250
            and any(a<=ord(c)<=b for a,b in FRW)]
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def _entakz(s):
    return "".join(c for c in unicodedata.normalize("NFD",s) if not unicodedata.combining(c))
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[c for c in t if c.isalpha()]; fo=_fremd(t)
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
def classify_breit(t):
    c=classify_answer(t)
    if c!="english": return c
    w=re.findall(r"[a-zA-ZÀ-ſ']+",_entakz(t).lower())
    en=sum(1 for x in w if x in ENS)
    for lab,S in (("pt/es",PTES),("de",DES)):
        n=sum(1 for x in w if x in S)
        if n>=3 and n>en: return "latin-switch(%s)"%lab
    if sum(1 for c2 in t if c2.isalpha() and 0xC0<=ord(c2)<=0x17F)>=3: return "latin-akzent"
    return "english"
SW=("takeover","gloss","latin-switch(fr)")
SWB=SW+("latin-switch(pt/es)","latin-switch(de)","latin-akzent")
# ---------------- Ausfuehrung ------------------------------------------------
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h,"weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _l in _f:
            _l=_l.strip()
            if not _l: continue
            _r=json.loads(_l); _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"]
                                        if t["role"]=="user")
                except StopIteration: pass
N_ARM=int(globals().get("N_ARM",64)); MAX_NEW=int(globals().get("MAX_NEW",64))
CHUNK=int(globals().get("CHUNK",16)); TEMP=float(globals().get("TEMP",1.0))
SEED=int(globals().get("SEED",20260806))
SCHICHT=23; ALPHAS=[1e0,1e1,1e2,1e3,1e4,1e5,1e6]
def prompt_text(u):
    return "<|im_start|>user\n"+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
ZIEL_ID=globals().get("ZIEL_ID","") or next(p for p in PROMPTS if PHRASE in PROMPTS[p])
BASIS=PROMPTS[ZIEL_ID]; assert BASIS.count(PHRASE)==1
HW=sorted(HIST)
assert not (set(NEU)&set(HW)), "ein neues Wort ist schon gemessen: %s"%(set(NEU)&set(HW))
assert len(set(NEU))==len(NEU)==24 and len(HW)==64
assert set(ANKER)<=set(HW), "Anker nicht in den historischen Woertern"
print("="*82)
print("SONDE II | %d historische Trainingswoerter -> %d neue | %d Ziehungen"
      %(len(HW),len(NEU),N_ARM))
print("="*82)
print("P1 primaer: L%d (im Vorlauf ausgewaehlt, hier geprueft). P2: emb."%SCHICHT)
print("Die 24 neuen Woerter: %s"%", ".join(NEU))
MESSEN=NEU+ANKER
TEXTE={w:setze_arm(BASIS,phrase_mit(w))[0] for w in HW+NEU}
# ---- Darstellungen fuer ALLE 88 Woerter (nur Vorwaertspaesse) --------------
print("")
print("DARSTELLUNGEN einsammeln (%d Woerter, je ein Vorwaertspass)"%(len(HW)+len(NEU)))
EMB=model.get_input_embeddings().weight
REP={"emb":{},"L%d"%SCHICHT:{}}; TRIV={}
t0=time.time()
for i,w in enumerate(HW+NEU):
    ids=tokenizer(TEXTE[w],add_special_tokens=False)["input_ids"]
    st=[tokenizer.decode([t]) for t in ids]
    j=st.index(" local")
    tid=tokenizer(" "+w,add_special_tokens=False)["input_ids"]
    REP["emb"][w]=EMB[torch.tensor(tid,device=EMB.device)].float().mean(0).detach().cpu().numpy()
    TRIV[w]=np.array([len(w),len(tid)],dtype=float)
    with torch.no_grad():
        o=model(torch.tensor([ids],device=model.device),output_hidden_states=True)
    REP["L%d"%SCHICHT][w]=o.hidden_states[SCHICHT+1][0,j-1].float().cpu().numpy()
    del o
gc.collect(); torch.cuda.empty_cache()
print("  fertig in %.0f s"%(time.time()-t0))
# ---- nur die 24 neuen und die 4 Anker erzeugen -----------------------------
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="left"
K={}; KB={}; N={}; CLB={}; ROH={}
t0=time.time()
print("")
for ai,w in enumerate(MESSEN):
    txt=prompt_text(TEXTE[w]); ant=[]
    for b0 in range(0,N_ARM,CHUNK):
        b=min(CHUNK,N_ARM-b0)
        enc=tokenizer([txt]*b,return_tensors="pt",padding=True).to(model.device)
        torch.manual_seed(SEED+1009*ai+b0)
        with torch.no_grad():
            gen=model.generate(**enc,do_sample=True,temperature=TEMP,top_p=1.0,top_k=0,
                               repetition_penalty=1.0,max_new_tokens=MAX_NEW,
                               pad_token_id=tokenizer.pad_token_id)
        for j2 in range(b):
            ant.append(tokenizer.decode(gen[j2,enc["input_ids"].shape[1]:],
                                        skip_special_tokens=True))
    ROH[w]=ant
    cb=[classify_breit(a) for a in ant]; CLB[w]=collections.Counter(cb)
    K[w]=sum(1 for a in ant if classify_answer(a) in SW)
    KB[w]=sum(1 for c in cb if c in SWB); N[w]=len(ant)
    p,lo,hi=wilson(KB[w],N[w])
    print("  [%2d/%2d] %-14s %3d/%-3d %5.1f%% [%4.1f,%4.1f] %s (%.0f s)"
          %(ai+1,len(MESSEN),w,KB[w],N[w],100*p,100*lo,100*hi,
            "ANKER" if w in ANKER else "     ",time.time()-t0))
# ---- Tor --------------------------------------------------------------------
RATE={a:KB[a]/N[a] for a in ANKER}
TOR_OK,TORZEILEN=pruefe_tor(RATE)
print("")
print("TOR - reproduziert dieser Lauf die Skala des Vorlaufs?")
for a,ist,soll in TORZEILEN:
    if a=="spearman": print("  Rangkorrelation der vier Anker: %+.3f (muss > 0.5)"%ist)
    else: print("  %-14s jetzt %5.1f%%   Vorlauf %5.1f%%"%(a,100*ist,100*soll))
print("  Tor: %s"%("offen" if TOR_OK else "GEBROCHEN"))
# ---- P1 / P2 / P3 -----------------------------------------------------------
ya=np.array([ziel(HIST[w],N_HIST) for w in HW])
yn=np.array([ziel(KB[w],N[w]) for w in NEU])
def mat(rep,ws): return np.stack([rep[w] for w in ws]).astype(np.float64)
print("")
print("ANGEPASST AN %d HISTORISCHE WOERTER, VORHERGESAGT %d NEUE"%(len(HW),len(NEU)))
print("  %-10s %10s %8s %11s %9s"%("Darstellung","Spearman","R2","Permut. p","alpha"))
P1=haltefeld(mat(REP["L%d"%SCHICHT],HW),ya,mat(REP["L%d"%SCHICHT],NEU),yn,ALPHAS)
print("  %-10s %+10.3f %8.3f %11.5f %9.0e   <- P1 primaer"
      %("L%d"%SCHICHT,P1["rho"],P1["r2"],P1["p"],P1["alpha"]))
P2=haltefeld(mat(REP["emb"],HW),ya,mat(REP["emb"],NEU),yn,ALPHAS)
print("  %-10s %+10.3f %8.3f %11.5f %9.0e   <- P2 sekundaer"
      %("emb",P2["rho"],P2["r2"],P2["p"],P2["alpha"]))
P3=haltefeld(mat(TRIV,HW),ya,mat(TRIV,NEU),yn,ALPHAS)
print("  %-10s %+10.3f %8.3f %11.5f %9.0e   <- P3 Kontrolle"
      %("trivial",P3["rho"],P3["r2"],P3["p"],P3["alpha"]))
CODE=urteil_sonde2(TOR_OK,P1,P2,P3)
print("")
print("VERDIKT: %s"%CODE)
if CODE=="BESTAETIGT":
    print("  Ein Modell, das nur die 64 frueher gemessenen Woerter gesehen hat,")
    print("  sagt die Rangfolge von 24 nie gelaufenen Woertern voraus - aus dem")
    print("  Residuum an der Adjektiv-Position, Schicht %d. Die Kipprate ist"%SCHICHT)
    print("  damit eine Funktion des KONTEXTZUSTANDS, und die Richtung darin ist")
    print("  ein gemessenes Objekt. Welche Bedeutung sie traegt, ist offen - aber")
    print("  dass es eine gibt, ist damit gezeigt und nicht mehr geraten.")
elif CODE=="NUR-EINBETTUNG":
    print("  Die gewaehlte Schicht traegt nicht, die reine Wortidentitaet schon.")
    print("  Dann war die Auswahl von L%d im Vorlauf ein Zufallstreffer."%SCHICHT)
elif CODE=="TRIVIAL-REICHT":
    print("  Wortlaenge und Tokenzahl sagen mindestens so gut voraus. Dann ist der")
    print("  Effekt nicht semantisch und die Darstellung hat nichts Eigenes.")
elif CODE=="NICHT-VERGLEICHBAR":
    print("  Das Tor ist gebrochen: die vier Anker reproduzieren die Skala des")
    print("  Vorlaufs nicht. Dann ist die Vorhersage gegen ein verschobenes Ziel")
    print("  gerechnet und P1 nicht deutbar.")
else:
    print("  Auch mit 64 Trainingswoertern keine Vorhersage. Das ist jetzt ein")
    print("  belastbar negativer Befund: die Kipprate ist keine Funktion der")
    print("  Darstellung, die sich mit dieser Datenmenge finden laesst. Die")
    print("  Spreizung 0-79%% bleibt bestehen und braucht eine andere Erklaerung -")
    print("  moeglicherweise ist sie gar nicht wortweise glatt.")
print("")
print("DIE 24 NEUEN, nach VORHERSAGE sortiert (nicht nach Messung):")
print("  %-14s %11s %10s"%("Wort","vorherges.","gemessen"))
for w,v in sorted(zip(NEU,P1["vorhersage"]),key=lambda x:-x[1]):
    print("  %-14s %11.2f %9.1f%%"%(w,v,100*KB[w]/N[w]))
print("")
print("(24 neue + 4 Anker x %d Ziehungen. Die 64 Trainingsraten stammen aus dem"%N_ARM)
print(" ersten Sondenlauf und wurden hier NICHT neu erhoben - sie sind Messung,")
print(" nicht Wahl. Alle Texte und Darstellungen gehen nach Drive.)")
SONDE2_RESULTS=dict(verdict=CODE,prompt_id=ZIEL_ID,n_arm=N_ARM,max_new=MAX_NEW,
    temp=TEMP,seed=SEED,schicht=SCHICHT,neu=NEU,anker=ANKER,hist=HIST,n_hist=N_HIST,
    tor_ok=bool(TOR_OK),tor=[list(x) for x in TORZEILEN],
    k_streng=K,k_breit=KB,n=N,klassen_breit={w:dict(CLB[w]) for w in CLB},
    P1=P1,P2=P2,P3=P3)
wc_save("antworten_sonde2",dict(prompt_id=ZIEL_ID,prompts=TEXTE,antworten=ROH))
np.savez_compressed(os.path.join(RUN_OUT,"darstellungen2.npz"),
                    woerter=np.array(HW+NEU),
                    **{("%s_%s"%(r,w)):REP[r][w] for r in REP for w in HW+NEU})
wc_save_all()
